In [ ]:
def process_order(text, customer_id="C001", verbose=True):
    # 1) 주문 분석: 자연어 → OrderRequest (제품명·수량·우선순위)
    req = analyze_order(text)

    # 2) 제품 식별: resolve_product(문자열 매칭)·get_product(코드 검증) 사용
    ident = identify_product(text, product_hint=req.product_name)
    if ident["product"] is None:
        # 못 찾음 → 후보 목록과 '되묻기 필요' 신호만 반환 (실제 되묻기는 09가 담당)
        return {"need_clarification": True, "candidates": ident["candidates"], ...}
    pid = ident["product"]["product_id"]

    # 3) 주문 생성 — 05: 주문을 DB에 등록하고 주문번호·금액 반환
    order = tools.create_order(pid, req.quantity, customer_id=customer_id, ...)

    # 4) 재고 조회 — 05: 현재 재고와 부족 수량(shortage) 계산
    inv = tools.get_inventory(pid, required_qty=req.quantity)

    # 5) 처리 방향 결정 — 06: 재고·자재를 따져 ship/produce/purchase 판단
    #    (이 시스템에서 '무엇을 할지'를 정하는 핵심. Agent는 실행만 함)
    decision = rules.decide_order(pid, req.quantity)

    # 6) 액션 실행
    if decision["action"] == "ship":
        tools.update_order_status(oid, "shipped")     # 05: 상태→shipped
    else:
        for p in decision["purchase_plan"]:
            po = tools.create_purchase_order(...)     # 05: 부족 자재 발주 생성
            po_writer.issue_purchase_order(po, ...)   # 12: 발주서(txt) 작성
        tools.create_production_order(pid, ...)        # 05: 생산계획 생성
        tools.update_order_status(oid, "confirmed")   # 05: 상태→confirmed